# Assignment 2 - Task 3: LoRA Finetuning
신규 데이터집합을 이용한 IP2P 모델의 LoRA Finetuning 구현

## 실험 방법
- **Method 1**: 전체 UNet에 LoRA 적용
- **Method 2**: 특정 블록만 선택하여 LoRA 적용
  - 다양한 블록 조합(`SELECTED_BLOCKS`)을 실험하여 최적 조합 찾기
  - Training Time과 결과 품질을 비교하여 하나를 선택

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

work_dir = '/content/drive/MyDrive/machine_learning_hw10/IP2P_LoRA_FT'
sys.path.append(work_dir)

model_dir = os.path.join(work_dir, 'saved_models')
os.makedirs(model_dir, exist_ok=True)

LORA_METHOD1_PATH = os.path.join(model_dir, "lora_method1.pt")
LORA_METHOD2_PATH = os.path.join(model_dir, "lora_method2.pt") # method2 실험 실행 전 selected blocks 조합에 따라 구별 가능하도록 파일 이름 수정

In [ ]:
!pip install -q diffusers transformers accelerate datasets pillow matplotlib tqdm

In [ ]:
from lora_utils import LoRALinearLayer, LoRALayer, apply_lora_to_unet_full, apply_lora_to_unet_selective

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from diffusers import DDPMScheduler, AutoencoderKL, UNet2DConditionModel, StableDiffusionInstructPix2PixPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from datasets import load_dataset
from torchvision import transforms
from tqdm.auto import tqdm
import numpy as np
import time
import matplotlib.pyplot as plt
import random

device = torch.device("cuda")
print(f"Device: {device}")

In [ ]:
# 하이퍼파라미터
MODEL_ID = "timbrooks/instruct-pix2pix"
DATASET_NAME = "instruction-tuning-sd/cartoonization"

LORA_RANK = 8
LORA_ALPHA = 8
TRAIN_TEST_SPLIT = 0.96
RESOLUTION = 256
SEED = 42

NUM_EPOCHS = 5
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5

# Method 2용: 다양한 조합을 실험해보세요
# up, mid, down 블록들 중 한가지 또는 두가지 조합으로 입력을 바꾸어 실험
SELECTED_BLOCKS = ["up", "mid"]

print(f"Rank: {LORA_RANK}, Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}")
print(f"Method 2 blocks: {SELECTED_BLOCKS}")

Rank: 8, Epochs: 5, Batch: 4
Method 2 blocks: ['up', 'mid']


## 데이터셋 준비

In [ ]:
# 데이터셋 로드 및 분할
dataset = load_dataset(DATASET_NAME)
total_samples = len(dataset['train'])
num_train = int(total_samples * TRAIN_TEST_SPLIT)

shuffled = dataset['train'].shuffle(seed=SEED)
train_dataset = shuffled.select(range(num_train))

print(f"Training samples: {num_train}")

In [ ]:
# 데이터 전처리
def convert_to_np(image, resolution):
    image = image.convert("RGB").resize((resolution, resolution))
    return np.array(image).transpose(2, 0, 1)

tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")

def tokenize_captions(captions, tokenizer):
    inputs = tokenizer(captions, max_length=tokenizer.model_max_length,
                      padding="max_length", truncation=True, return_tensors="pt")
    return inputs.input_ids

train_transforms = transforms.Compose([transforms.RandomCrop(RESOLUTION)])

def preprocess_fn(examples):
    original_images = np.stack([convert_to_np(img, RESOLUTION) for img in examples['original_image']])
    edited_images = np.stack([convert_to_np(img, RESOLUTION) for img in examples['cartoonized_image']])

    images = np.concatenate([original_images, edited_images])
    images = torch.tensor(images).float()
    images = 2 * (images / 255) - 1
    images = train_transforms(images)

    original_images, edited_images = images.chunk(2)
    examples["original_pixel_values"] = original_images
    examples["edited_pixel_values"] = edited_images
    examples["input_ids"] = tokenize_captions(list(examples['edit_prompt']), tokenizer)
    return examples

train_dataset_processed = train_dataset.with_transform(preprocess_fn)

def collate_fn(examples):
    return {
        "original_pixel_values": torch.stack([ex["original_pixel_values"] for ex in examples]),
        "edited_pixel_values": torch.stack([ex["edited_pixel_values"] for ex in examples]),
        "input_ids": torch.stack([ex["input_ids"] for ex in examples]),
    }

train_dataloader = DataLoader(train_dataset_processed, batch_size=BATCH_SIZE,
                             shuffle=True, collate_fn=collate_fn, num_workers=2)

print(f"Dataloader ready with {len(train_dataloader)} batches")

---
## Method 1: Full UNet LoRA Finetuning

In [ ]:
print("=" * 70)
print("Method 1: Full UNet LoRA Finetuning")
print("=" * 70)

# 모델 로드
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# LoRA 적용
lora_layers_1, trainable_params_1 = apply_lora_to_unet_full(unet, rank=LORA_RANK, alpha=LORA_ALPHA)
print(f"Applied {len(lora_layers_1)} LoRA layers")
print(f"Trainable parameters: {trainable_params_1:,}")

# Optimizer
optimizer = torch.optim.AdamW([p for lora in lora_layers_1 for p in lora.lora.parameters()], lr=LEARNING_RATE)

# 학습
start_time = time.time()
unet.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for step, batch in enumerate(progress_bar):
        with torch.no_grad():
            latents_orig = vae.encode(batch["original_pixel_values"].to(device)).latent_dist.sample() * 0.18215
            latents_edit = vae.encode(batch["edited_pixel_values"].to(device)).latent_dist.sample() * 0.18215
            encoder_hidden_states = text_encoder(batch["input_ids"].to(device))[0]

        noise = torch.randn_like(latents_edit)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                 (latents_edit.shape[0],), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents_edit, noise, timesteps)
        latent_model_input = torch.cat([noisy_latents, latents_orig], dim=1)

        model_pred = unet(latent_model_input, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(model_pred, noise, reduction="mean")
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        progress_bar.set_postfix({"loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})

    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_dataloader):.4f}")

train_time_1 = time.time() - start_time
print(f"Training completed in {train_time_1/60:.2f} minutes")

# 모델 저장
lora_state_dict = {f"lora_{i}_A": lora.lora.lora_A.data.cpu() for i, lora in enumerate(lora_layers_1)}
lora_state_dict.update({f"lora_{i}_B": lora.lora.lora_B.data.cpu() for i, lora in enumerate(lora_layers_1)})
lora_state_dict["trainable_params"] = trainable_params_1
lora_state_dict["train_time"] = train_time_1
torch.save(lora_state_dict, LORA_METHOD1_PATH)
print(f"Saved to {LORA_METHOD1_PATH}")

del unet, vae, text_encoder, optimizer
torch.cuda.empty_cache()

### Method 1 Inference (결과 확인)

In [ ]:
# 테스트 샘플 준비
num_test = int(total_samples * (1 - TRAIN_TEST_SPLIT))
test_dataset = shuffled.select(range(num_train, total_samples))

random.seed()
test_idx_1 = random.randint(0, len(test_dataset) - 1)
test_sample_1 = test_dataset[test_idx_1]
test_image_1 = test_sample_1['original_image']
test_prompt_1 = test_sample_1['edit_prompt']

print(f"Test sample {test_idx_1}: {test_prompt_1}")

In [ ]:
# Method 1 모델 로드 및 Inference
print("Loading Method 1 model for inference...")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device)

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_layers_1_infer, _ = apply_lora_to_unet_full(unet, rank=LORA_RANK, alpha=LORA_ALPHA)
lora_state_1 = torch.load(LORA_METHOD1_PATH, map_location='cpu')

for i, lora_layer in enumerate(lora_layers_1_infer):
    lora_layer.lora.lora_A.data = lora_state_1[f"lora_{i}_A"].to(device)
    lora_layer.lora.lora_B.data = lora_state_1[f"lora_{i}_B"].to(device)

pipe_1 = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    MODEL_ID, unet=unet, text_encoder=text_encoder, vae=vae,
    torch_dtype=torch.float32, safety_checker=None
).to(device)

with torch.no_grad():
    output_1 = pipe_1(
        test_prompt_1, image=test_image_1, num_inference_steps=20,
        image_guidance_scale=1.5, guidance_scale=7.0
    ).images[0]

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(test_image_1)
axes[0].set_title('Original', fontsize=12, weight='bold')
axes[0].axis('off')
axes[1].imshow(output_1)
axes[1].set_title(f'Method 1 Output\n(Time: {train_time_1/60:.1f}min)', fontsize=12, weight='bold', color='blue')
axes[1].axis('off')
plt.suptitle(f'Prompt: "{test_prompt_1}"', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("Method 1 Summary")
print("="*70)
print(f"Trainable params: {trainable_params_1:,} ({trainable_params_1/1e6:.2f}M)")
print(f"Training time: {train_time_1/60:.2f} minutes")
print("="*70)

del pipe_1, unet, vae, text_encoder
torch.cuda.empty_cache()

---
## Method 2: Selective Block LoRA Finetuning

In [ ]:
print("=" * 70)
print("Method 2: Selective Block LoRA Finetuning")
print(f"Selected blocks: {SELECTED_BLOCKS}")
print("=" * 70)

# 모델 로드
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# LoRA 적용 (선택된 블록만)
lora_layers_2, trainable_params_2 = apply_lora_to_unet_selective(
    unet, rank=LORA_RANK, alpha=LORA_ALPHA, selected_blocks=SELECTED_BLOCKS
)
print(f"Applied {len(lora_layers_2)} LoRA layers to blocks: {SELECTED_BLOCKS}")
print(f"Trainable parameters: {trainable_params_2:,}")

# Optimizer
optimizer = torch.optim.AdamW([p for lora in lora_layers_2 for p in lora.lora.parameters()], lr=LEARNING_RATE)

# 학습
start_time = time.time()
unet.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for step, batch in enumerate(progress_bar):
        with torch.no_grad():
            latents_orig = vae.encode(batch["original_pixel_values"].to(device)).latent_dist.sample() * 0.18215
            latents_edit = vae.encode(batch["edited_pixel_values"].to(device)).latent_dist.sample() * 0.18215
            encoder_hidden_states = text_encoder(batch["input_ids"].to(device))[0]

        noise = torch.randn_like(latents_edit)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                 (latents_edit.shape[0],), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents_edit, noise, timesteps)
        latent_model_input = torch.cat([noisy_latents, latents_orig], dim=1)

        model_pred = unet(latent_model_input, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(model_pred, noise, reduction="mean")
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        progress_bar.set_postfix({"loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})

    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_dataloader):.4f}")

train_time_2 = time.time() - start_time
print(f"Training completed in {train_time_2/60:.2f} minutes")

# 모델 저장
lora_state_dict = {f"lora_{i}_A": lora.lora.lora_A.data.cpu() for i, lora in enumerate(lora_layers_2)}
lora_state_dict.update({f"lora_{i}_B": lora.lora.lora_B.data.cpu() for i, lora in enumerate(lora_layers_2)})
lora_state_dict["trainable_params"] = trainable_params_2
lora_state_dict["train_time"] = train_time_2
torch.save(lora_state_dict, LORA_METHOD2_PATH)
print(f"Saved to {LORA_METHOD2_PATH}")

del unet, vae, text_encoder, optimizer
torch.cuda.empty_cache()

### Method 2 Inference (결과 확인)

In [ ]:
# 테스트 샘플 준비
random.seed()
test_idx_2 = random.randint(0, len(test_dataset) - 1)
test_sample_2 = test_dataset[test_idx_2]
test_image_2 = test_sample_2['original_image']
test_prompt_2 = test_sample_2['edit_prompt']

print(f"Test sample {test_idx_2}: {test_prompt_2}")

In [ ]:
# Method 2 모델 로드 및 Inference
print(f"Loading Method 2 model (blocks: {SELECTED_BLOCKS}) for inference...")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device)

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_layers_2_infer, _ = apply_lora_to_unet_selective(
    unet, rank=LORA_RANK, alpha=LORA_ALPHA, selected_blocks=SELECTED_BLOCKS
)
lora_state_2 = torch.load(LORA_METHOD2_PATH, map_location='cpu')

for i, lora_layer in enumerate(lora_layers_2_infer):
    lora_layer.lora.lora_A.data = lora_state_2[f"lora_{i}_A"].to(device)
    lora_layer.lora.lora_B.data = lora_state_2[f"lora_{i}_B"].to(device)

pipe_2 = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    MODEL_ID, unet=unet, text_encoder=text_encoder, vae=vae,
    torch_dtype=torch.float32, safety_checker=None
).to(device)

with torch.no_grad():
    output_2 = pipe_2(
        test_prompt_2, image=test_image_2, num_inference_steps=20,
        image_guidance_scale=1.5, guidance_scale=7.0
    ).images[0]

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(test_image_2)
axes[0].set_title('Original', fontsize=12, weight='bold')
axes[0].axis('off')
axes[1].imshow(output_2)
axes[1].set_title(f'Method 2 Output ({SELECTED_BLOCKS})\n(Time: {train_time_2/60:.1f}min)',
                 fontsize=12, weight='bold', color='green')
axes[1].axis('off')
plt.suptitle(f'Prompt: "{test_prompt_2}"', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print(f"Method 2 Summary (Selected blocks: {SELECTED_BLOCKS})")
print("="*70)
print(f"Trainable params: {trainable_params_2:,} ({trainable_params_2/1e6:.2f}M)")
print(f"Training time: {train_time_2/60:.2f} minutes")
print("="*70)

del pipe_2, unet, vae, text_encoder
torch.cuda.empty_cache()